# Fine-Tuning a Self-Hosted Mistral for Grounded, Refusal-Aware Answers

This notebook documents an additional optimisation experiment for the generation stage of the
retrieval-augmented pipeline. The generator comparison in Section 4.9 exposed a trade-off between
fluency and faithfulness: some models answered fluently while drifting away from the retrieved
German context, and others refused so frequently that they lost practical value. The experiment
examines whether a locally hosted Mistral-7B, instruction-tuned on the project's held-out data,
can be moved towards the safer region of that trade-off, that is, answering strictly from the
AWMF context and returning an explicit refusal when the answer is absent.

Two constraints preserve validity. Training uses only the MedQA-derived questions that lie
outside the 200-question evaluation subset of Chapter 4, so the evaluation set remains untouched
and no leakage occurs. The model, the embedder, and the evaluation run locally on the Colab GPU,
and faithfulness is first approximated with deterministic, judge-free proxies, so the initial
procedure does not depend on an external scoring service. The result is not assumed in advance.
Fine-tuning on a few hundred examples may overfit rather than improve, and the notebook measures
the fine-tuned model against the unmodified base model under identical conditions.

## 1. Environment

The notebook targets a Colab GPU runtime. A T4 is sufficient for 4-bit QLoRA, and an A100 is
faster. The installation below builds the training stack around Unsloth, which fits a 7B model
into a single consumer GPU.

In [ ]:
%%capture
# QLoRA training stack (Unsloth fits a 7B model into roughly 15 GB of VRAM).
!pip install -q -U unsloth
!pip install -q -U trl peft accelerate bitsandbytes datasets
# Retrieval and judge-free evaluation.
!pip install -q -U sentence-transformers faiss-cpu

## 2. Configuration

The configuration references the files produced earlier in the project. Column names are kept
configurable because the benchmark CSV may label its fields differently; the first data cell
prints the available columns.

In [ ]:
CONFIG = {
    # data
    "BENCHMARK_CSV": "GP_Top10_Bilingual_Open_EndedQ.csv",  # the full 715-row bilingual benchmark
    "EVAL200_CSV":   "eval_200_subset.csv",                 # the 200 questions used in Chapter 4 (held out of training)
    "AWMF_CHUNKS_CSV": "awmf_chunks.csv",                   # the AWMF chunks (one text column)

    # column names as they appear in the benchmark CSV
    "ID_COL":     "id",
    "Q_EN_COL":   "question_en",
    "Q_DE_COL":   "question_de",
    "ANSWER_COL": "answer",
    "CHUNK_TEXT_COL": "page_content",

    # models
    "MODEL":    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "EMBEDDER": "BAAI/bge-m3",

    # retrieval and training
    "TOP_K":    5,
    "MAX_SEQ":  2048,
    "EPOCHS":   2,
    "EVAL_N":   40,      # number of held-out evaluation questions to score
    "OUTPUT_DIR": "mistral_grounded_lora",

    # RAGAS (optional, comparable to Chapter 4). Generation is self-hosted; only the judge uses an API.
    "GROUND_TRUTH_CSV": None,                          # optional corpus-grounded references (enables context recall/precision)
    "GT_Q_COL":   "question_en",
    "GT_REF_COL": "reference",
    "JUDGE_MODEL":    "openai/gpt-4o-mini",            # same judge family as Chapter 4
    "JUDGE_BASE_URL": "https://openrouter.ai/api/v1",  # a local OpenAI-compatible endpoint may be used instead
}

## 3. Held-out split

The training set is defined by subtraction: the benchmark rows whose identifier is not present in
the 200-question evaluation subset. When no stable identifier column exists, the fallback matches
on the English question text.

In [ ]:
import pandas as pd

bench = pd.read_csv(CONFIG["BENCHMARK_CSV"])
print("benchmark rows:", len(bench))
print("columns:", bench.columns.tolist())
bench.head(2)

In [ ]:
eval200 = pd.read_csv(CONFIG["EVAL200_CSV"])
id_col, qen = CONFIG["ID_COL"], CONFIG["Q_EN_COL"]

if id_col in bench.columns and id_col in eval200.columns:
    eval_keys = set(eval200[id_col].astype(str))
    held = bench[~bench[id_col].astype(str).isin(eval_keys)].copy()
    eval_df = bench[bench[id_col].astype(str).isin(eval_keys)].copy()
else:
    # Fallback: identify the evaluation rows by their English question text.
    eval_keys = set(eval200[qen].astype(str).str.strip())
    mask = bench[qen].astype(str).str.strip().isin(eval_keys)
    held, eval_df = bench[~mask].copy(), bench[mask].copy()

print(f"held-out for training: {len(held)}   |   evaluation subset (untouched): {len(eval_df)}")

## 4. German AWMF retrieval index (self-hosted BGE-M3 and FAISS)

Both dataset construction and evaluation require the context that an English query retrieves at
inference time. The index is built once in memory with the multilingual BGE-M3 embedder, which
removes any dependency on the production database. The chunk text can alternatively be read from
the Neon langchain_pg_embedding table; the retrieval function only requires a list of German
chunk strings.

In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

chunks = pd.read_csv(CONFIG["AWMF_CHUNKS_CSV"])[CONFIG["CHUNK_TEXT_COL"]].dropna().astype(str).tolist()
print("AWMF chunks:", len(chunks))

embedder = SentenceTransformer(CONFIG["EMBEDDER"], device="cuda")
chunk_emb = embedder.encode(chunks, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb.astype("float32"))

def retrieve(query, k=CONFIG["TOP_K"]):
    v = embedder.encode([query], normalize_embeddings=True).astype("float32")
    _, idx = index.search(v, k)
    return [chunks[i] for i in idx[0]]

## 5. Instruction dataset (grounded answers and explicit refusals)

Each held-out question is paired with the context retrieved for its English form. The target
depends on whether that context supports the reference answer. When a sufficient share of the
reference answer's salient terms appears in the retrieved context, the target is the grounded
reference answer. Otherwise the target is the exact refusal token NOT_IN_CORPUS. Supervising both
cases is the purpose of the experiment: the refusal examples teach the model to withhold content
when the guidelines do not cover the question. The coverage rule below is a deliberately simple
lexical heuristic, chosen for transparency and reproducibility rather than exactness.

In [ ]:
import re

STRICT = (
    "You are a clinical decision-support assistant for German general practice. "
    "Answer strictly and only from the CONTEXT below, which is drawn from the German AWMF "
    "guidelines. Do not add facts that are not in the context. If the context does not contain "
    "the answer, reply with exactly NOT_IN_CORPUS and nothing else."
)

def salient(text):
    return set(re.findall(r"[A-Za-zAeOeUeaeoeuess]{4,}", str(text).lower()))

def is_answerable(answer, context, threshold=0.35):
    toks = salient(answer)
    if not toks:
        return False
    ctx = context.lower()
    covered = sum(1 for t in toks if t in ctx)
    return covered / len(toks) >= threshold

records = []
for _, row in held.iterrows():
    q_en = str(row[CONFIG["Q_EN_COL"]]).strip()
    ans  = str(row[CONFIG["ANSWER_COL"]]).strip()
    ctx  = "\n\n".join(retrieve(q_en))
    target = ans if is_answerable(ans, ctx) else "NOT_IN_CORPUS"
    records.append({
        "user":   f"{STRICT}\n\nCONTEXT:\n{ctx}\n\nQUESTION: {q_en}",
        "target": target,
    })

n_refuse = sum(1 for r in records if r["target"] == "NOT_IN_CORPUS")
print(f"training examples: {len(records)}  (grounded: {len(records)-n_refuse}, refusals: {n_refuse})")

The salient() pattern uses ASCII placeholders for the German umlauts so the code remains portable
across editors. If the corpus text preserves the original umlauts, the character class can include
them directly; retrieval and training are unaffected either way.

## 6. Load Mistral-7B in 4-bit and attach LoRA adapters

The LoRA update matrices are zero-initialised, so immediately after this cell the adapted model
behaves identically to the base model. Section 7 relies on this property: the base scores are
measured on the same object before any training step runs.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = CONFIG["MODEL"],
    max_seq_length = CONFIG["MAX_SEQ"],
    load_in_4bit = True,
    dtype = None,
)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0.0,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth", random_state = 42,
)

## 7. Judge-free evaluation harness

Two deterministic signals stand in for an external RAGAS judge. Refusal behaviour checks whether
out-of-corpus questions produce NOT_IN_CORPUS and in-corpus questions do not, which addresses the
hallucination problem directly. Lexical grounding measures, for the answers the model does give,
how much of the answer vocabulary is present in the retrieved context; a higher value indicates
an answer closer to the source. The same harness scores the base model, with adapters still zero,
and later the fine-tuned model.

In [ ]:
import torch

def generate(user_text, max_new_tokens=256):
    msgs = [{"role": "user", "content": user_text}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def grounding(answer, context):
    toks = salient(answer)
    if not toks:
        return 0.0
    ctx = context.lower()
    return sum(1 for t in toks if t in ctx) / len(toks)

# Optional reference answers (question -> corpus-grounded reference) for RAGAS context recall/precision.
GT = {}
if CONFIG.get("GROUND_TRUTH_CSV"):
    _gt = pd.read_csv(CONFIG["GROUND_TRUTH_CSV"])
    GT = dict(zip(_gt[CONFIG["GT_Q_COL"]].astype(str).str.strip(), _gt[CONFIG["GT_REF_COL"]].astype(str)))

def evaluate(tag, n=CONFIG["EVAL_N"]):
    sample = eval_df.head(n)
    correct_refusal = over_refusal = grounded_sum = 0
    n_unans = n_ans = 0
    rows = []   # collected for the RAGAS run below
    for _, row in sample.iterrows():
        q_en = str(row[CONFIG["Q_EN_COL"]]).strip()
        gold = str(row[CONFIG["ANSWER_COL"]]).strip()
        ctxs = retrieve(q_en)
        ctx  = "\n\n".join(ctxs)
        out  = generate(f"{STRICT}\n\nCONTEXT:\n{ctx}\n\nQUESTION: {q_en}")
        refused = out.upper().startswith("NOT_IN_CORPUS")
        rows.append({"question": q_en, "answer": out, "contexts": ctxs, "ground_truth": GT.get(q_en)})
        if is_answerable(gold, ctx):
            n_ans += 1
            if refused: over_refusal += 1
            else:       grounded_sum += grounding(out, ctx)
        else:
            n_unans += 1
            if refused: correct_refusal += 1
    scores = {
        "model": tag,
        "refusal_recall (unanswerable)": round(correct_refusal / max(n_unans, 1), 3),
        "over_refusal (answerable)":     round(over_refusal / max(n_ans, 1), 3),
        "grounding (answerable)":        round(grounded_sum / max(n_ans - over_refusal, 1), 3),
        "n_ans/n_unans": f"{n_ans}/{n_unans}",
    }
    return scores, rows

base_scores, base_rows = evaluate("base (pre-training)")
base_scores

## 8. Supervised fine-tuning

A short QLoRA run over the held-out examples. Two epochs is generally sufficient for a few hundred
samples; longer runs tend to overfit the refusal token. Depending on the installed trl version,
two SFTConfig fields (dataset_text_field, max_seq_length) may need to move into the trainer call
instead, as the comment marks.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

def to_text(r):
    msgs = [
        {"role": "user", "content": r["user"]},
        {"role": "assistant", "content": r["target"]},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

train_ds = Dataset.from_list(records).map(to_text)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = CONFIG["EPOCHS"],
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        report_to = "none",
        output_dir = CONFIG["OUTPUT_DIR"],
        dataset_text_field = "text",   # trl>=0.11: keep here; older trl: pass to SFTTrainer(...)
        max_seq_length = CONFIG["MAX_SEQ"],
    ),
)
trainer.train()

## 9. Re-score the fine-tuned model and compare

The same harness runs on the same evaluation questions with the same retrieved context. Only the
adapter weights differ.

In [ ]:
tuned_scores, tuned_rows = evaluate("fine-tuned")
comparison = pd.DataFrame([base_scores, tuned_scores])
comparison

## 10. RAGAS evaluation (comparable to Chapter 4)

The proxies above provide a quick offline signal. The metric used throughout Chapter 4 is RAGAS,
which is applied here as well. The answers were already produced by the self-hosted model, so
RAGAS spends external calls only on the judge that scores the collected question, context, and
answer rows. This is a single scoring pass over the evaluation set rather than the repeated
development-loop calls of the earlier chapter, which keeps its cost modest.

Faithfulness and Answer Relevancy require no reference answer. Context Precision and Context
Recall are added when a corpus-grounded reference file is supplied through GROUND_TRUTH_CSV. A
fully offline run is possible by setting JUDGE_BASE_URL to a local OpenAI-compatible endpoint; a
smaller local judge is weaker than GPT-4o-mini, so those scores are indicative rather than
directly comparable to the chapter tables.

In [ ]:
!pip install -q ragas langchain-openai langchain-huggingface

import os
os.environ.setdefault("OPENROUTER_API_KEY", "")  # judge API key (environment variable)

from datasets import Dataset
from ragas import evaluate as ragas_evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

judge = LangchainLLMWrapper(ChatOpenAI(
    model = CONFIG["JUDGE_MODEL"], temperature = 0,
    base_url = CONFIG["JUDGE_BASE_URL"], api_key = os.environ["OPENROUTER_API_KEY"]))
judge_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name = CONFIG["EMBEDDER"]))

def run_ragas(rows):
    has_ref = all(r.get("ground_truth") for r in rows)
    data = {
        "question": [r["question"] for r in rows],
        "answer":   [r["answer"]   for r in rows],
        "contexts": [r["contexts"] for r in rows],
    }
    metrics = [faithfulness, answer_relevancy]
    if has_ref:
        data["ground_truth"] = [r["ground_truth"] for r in rows]
        metrics += [context_precision, context_recall]
    # Recent ragas versions rename these columns to user_input / response / retrieved_contexts / reference.
    return ragas_evaluate(Dataset.from_dict(data), metrics=metrics, llm=judge, embeddings=judge_emb)

print("base model RAGAS:")
print(run_ragas(base_rows))
print("fine-tuned model RAGAS:")
print(run_ragas(tuned_rows))

In [ ]:
# Persist the adapter so it can be reloaded or served later.
model.save_pretrained(f"{CONFIG['OUTPUT_DIR']}/lora")
tokenizer.save_pretrained(f"{CONFIG['OUTPUT_DIR']}/lora")
print("saved to", f"{CONFIG['OUTPUT_DIR']}/lora")

## 11. Interpretation

A favourable outcome is a higher refusal recall on out-of-corpus questions, indicating that the
model learns to emit NOT_IN_CORPUS, without a comparable rise in over-refusal on answerable
questions and without a fall in grounding. If refusal recall and over-refusal rise together, the
model has only learned to refuse more often, which is not the objective.

Two limitations qualify the result. The training set of a few hundred examples is small, so the
adapter can overfit the refusal token, which makes the over-refusal column the primary diagnostic.
The judge-free proxies are coarser than the RAGAS metrics used elsewhere. The definitive
comparison places this adapter in the generation stage of the full pipeline and re-runs the RAGAS
evaluation with the flagship judge, which yields the value comparable to the Mistral-Large row of
the generator evaluation.